# RNA-seq Gene Selection — SurvPath Option A

Filters the TCGA-BRCA RNA-seq matrix to the **4,999 SurvPath pathway genes** from `combine_signatures.csv`.

**Pipeline:**
1. Load the full RNA-seq parquet (from Google Drive)
2. Strip Ensembl `.version` suffixes
3. Map Ensembl IDs → HGNC gene symbols via `mygene` API
4. Intersect with SurvPath 4,999 genes
5. Save filtered matrix (`rnaseq_filtered_survpath.csv`)

In [5]:
# %pip install -q mygene pyarrow


In [6]:
import numpy as np
import pandas as pd

In [7]:
# Load dataset from local project folder
DATA_PATH = r'd:\UIU CSE\GitHub\FYDP-Project\tcga_brca_tri_modal_dataset.parquet'
SIGNATURES_PATH = r'd:\UIU CSE\GitHub\FYDP-Project\combine_signatures.csv'
OUTPUT_PATH = r'd:\UIU CSE\GitHub\FYDP-Project\rnaseq_filtered_survpath.csv'

df = pd.read_parquet(DATA_PATH)

# Keep only RNA-seq columns (after MUT_ANKRD12)
cut_idx = df.columns.get_loc('MUT_ANKRD12') + 1
df_rna = df.iloc[:, cut_idx:].copy()
print(f'RNA matrix shape: {df_rna.shape}')  # (1073, 60660)


RNA matrix shape: (1073, 60660)


In [8]:
import mygene

# 1. Strip Ensembl version suffixes (e.g. ENSG00000141510.15 -> ENSG00000141510)
base_ids = [col.split('.')[0] for col in df_rna.columns]
df_rna.columns = base_ids

# 2. Map Ensembl IDs to HGNC gene symbols via mygene API
mg = mygene.MyGeneInfo()
query_res = mg.querymany(
    list(set(base_ids)),
    scopes='ensembl.gene',
    fields='symbol',
    species='human',
    as_dataframe=True,
)

# 3. Build Ensembl -> Symbol mapping (drop unmapped)
id_to_symbol = query_res['symbol'].dropna().to_dict()
print(f'Mapped {len(id_to_symbol)} / {len(set(base_ids))} Ensembl IDs to gene symbols')

# 4. Rename columns and keep only mapped genes
df_rna = df_rna.rename(columns=id_to_symbol)
valid_symbols = set(id_to_symbol.values())
df_rna = df_rna[[c for c in df_rna.columns if c in valid_symbols]]

# 5. Handle duplicate gene symbols (average expression)
df_rna = df_rna.T.groupby(level=0).mean().T
print(f'RNA matrix after symbol mapping: {df_rna.shape}')

30 input query terms found dup hits:	[('ENSG00000234352', 2), ('ENSG00000262352', 3), ('ENSG00000280018', 2), ('ENSG00000287326', 2), ('E
1272 input query terms found no hit:	['ENSG00000287614', 'ENSG00000225393', 'ENSG00000258092', 'ENSG00000261445', 'ENSG00000270265', 'ENS


Mapped 45336 / 60616 Ensembl IDs to gene symbols
RNA matrix after symbol mapping: (1073, 44289)


In [9]:
# Option A: Use SurvPath combine_signatures.csv (4,999 genes)
# Upload combine_signatures.csv to Colab first (Files panel -> Upload)

sig_df = pd.read_csv(SIGNATURES_PATH)
survpath_genes = set(sig_df.values.ravel())
survpath_genes = {g for g in survpath_genes if isinstance(g, str) and g.strip()}
print(f'SurvPath unique genes: {len(survpath_genes)}')

# Keep only genes present in both SurvPath and your RNA-seq data
selected_genes = survpath_genes.intersection(set(df_rna.columns))
print(f'Genes in your dataset:               {len(df_rna.columns)}')
print(f'Matched SurvPath genes in your data: {len(selected_genes)}')
print(f'SurvPath genes NOT in your data:     {len(survpath_genes - set(df_rna.columns))}')

SurvPath unique genes: 4999
Genes in your dataset:               44289
Matched SurvPath genes in your data: 4906
SurvPath genes NOT in your data:     93


In [ ]:
# Sort gene names for consistent column ordering across runs
sorted_genes = sorted(list(selected_genes))
df_filtered = df_rna[sorted_genes]
print('Final shape:', df_filtered.shape)  # expected: (1073, ~4999)
df_filtered.head()

# # Save to local project folder
# df_filtered.to_csv(OUTPUT_PATH)
# print(f'Saved to: {OUTPUT_PATH}')

Final shape: (1073, 4906)
Saved to: d:\UIU CSE\GitHub\FYDP-Project\rnaseq_filtered_survpath.csv
